In [2]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
import matplotlib.pyplot as plt
import sys
from openpyxl import Workbook
import time

In [3]:
start_time = time.time()  # Record the start time

#Entry
tperiod = 12
# set a very negative initial lower bound            
thetaLB = 0

# Distribute evacuees to shelters

In [5]:
def allocate_demand(distance_file, demand_files, output_file, num_scenarios=100):
    
    # Load distance matrix (7 PODs, 21 Shelters, No Headers)
    distance_matrix = pd.read_excel(distance_file, header=None).iloc[:7, :21]

    # Compute inverse distance weights (avoid division by zero)
    inverse_distances = 1 / distance_matrix.replace(0, float("inf"))
    weights = inverse_distances.div(inverse_distances.sum(axis=1), axis=0)  # Normalize weights

    # Load demand data (7 files, each with 1048 scenarios & 6 time periods)
    demand_scenarios = [pd.read_csv(file, header=None) for file in demand_files]

    # Allocate demand for the specified number of scenarios
    allocated_demand = {}
    for scenario_idx in range(num_scenarios):
        # Extract demand for this scenario across all 7 PODs (6 time periods each)
        demand_df = pd.DataFrame({t: [demand_scenarios[pod].iloc[scenario_idx, t] for pod in range(7)] for t in range(6)})

        # Allocate demand based on distance weights
        allocation_matrix = {t: weights.multiply(demand_df[t], axis=0).sum(axis=0) for t in range(6)}

        # Convert to DataFrame where rows are shelters, columns are time periods
        allocated_demand[f"Scenario_{scenario_idx}"] = pd.DataFrame(allocation_matrix, index=range(21))

    # Save the results
    with pd.ExcelWriter(output_file) as writer:
        for scenario, df in allocated_demand.items():
            df.to_excel(writer, sheet_name=scenario)

    print(f"✅ Demand allocation completed. File saved: {output_file}")

# Example Usage:
distance_file = "distanceFull-podtoshelter.xlsx"
demand_files = [f"DP_{i}.csv" for i in range(7)]
output_file = "Shelter_Demand.xlsx"

allocate_demand(distance_file, demand_files, output_file, num_scenarios=100)

✅ Demand allocation completed. File saved: Shelter_Demand.xlsx


# Calculate PoD demand

In [7]:
def calculate_landfall_demand_fixed(wind_speed_file, pod_info_file, output_file):
    # Load the data
    wind_speed_df = pd.read_excel(wind_speed_file, header=None)
    pod_info_df = pd.read_excel(pod_info_file, sheet_name='Sheet1')

    # Read hurricane categories from column I (index 8)
    categories = wind_speed_df.iloc[:100, 8]

    # Define columns corresponding to each category
    category_columns = {
        "Category 1": "I*p_c (cat 1)",
        "Category 2": "I*p_c (cat 2)",
        "Category 3": "I*p_c (cat 3)",
        "Category 4": "I*p_c (cat 4)",
        "Category 5": "I*p_c (cat 5)"
    }

    # Get the column for '% 1- (Likely to evacuate)'
    evac_column = '% 1- (Likely to evacuate)'

    # Create an Excel writer to save the results
    with pd.ExcelWriter(output_file) as writer:
        for scenario_idx in range(100):
            # Get the category for this scenario
            category = categories.iloc[scenario_idx]

            # Check if the category is valid
            demand_column = category_columns.get(category, None)

            if demand_column is not None:
                # Extract the necessary data for calculation
                pod_demand_values = pod_info_df[demand_column]
                evacuation_factors = pod_info_df[evac_column]

                # Calculate demand: (I*p_c) * (% 1- Likely to evacuate)
                demand_values = pod_demand_values * evacuation_factors

            else:
                # If the category is N/A, set demand to 0
                demand_values = pd.Series(0, index=pod_info_df.index)

            # Ensure numeric data type
            demand_values = demand_values.astype(float)

            # Prepare the output DataFrame
            scenario_data = pd.DataFrame({
                'County': pod_info_df['County'],
                'total demand': demand_values
            })

            # Write to Excel sheet for this scenario
            sheet_name = f'Scenario_{scenario_idx}'
            scenario_data.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"✅ Corrected output file generated: {output_file}")

# Example usage of the function
wind_speed_file = 'wind_speed_err.xlsx'
pod_info_file = 'PoD location information.xlsx'
output_file = 'PoD_Demand_total.xlsx'

calculate_landfall_demand_fixed(wind_speed_file, pod_info_file, output_file)


✅ Corrected output file generated: PoD_Demand_total.xlsx


In [8]:
def distribute_demand_by_time(total_demand_file, output_file):
    # Load the total demand file
    total_demand_data = pd.ExcelFile(total_demand_file)

    # Define time periods and normal distribution parameters
    time_periods = [6, 7, 8, 9, 10, 11]
    mean = 8  # Peak at time period 8
    std_dev = 1  # Spread of the distribution

    # Calculate normalized weights using a normal distribution
    distribution_values = [np.exp(-((t - mean) ** 2) / (2 * (std_dev ** 2))) for t in time_periods]
    weights = np.array(distribution_values) / sum(distribution_values)  # Normalize weights to sum to 1

    # Create an Excel writer to save the results
    with pd.ExcelWriter(output_file) as writer:
        for sheet_name in total_demand_data.sheet_names:
            # Read the data for the current scenario
            scenario_data = pd.read_excel(total_demand_data, sheet_name=sheet_name)

            # Apply the distribution weights to the total demand for each county
            distributed_demands = {
                f'Time_{t}': scenario_data['total demand'] * weight for t, weight in zip(time_periods, weights)
            }

            # Prepare the output DataFrame
            output_data = scenario_data[['County']].copy()
            for period, demand in distributed_demands.items():
                output_data[period] = demand

            # Write the output data to the Excel sheet
            output_data.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"✅ Distributed demand file generated: {output_file}")

# Example usage of the function
total_demand_file = 'PoD_Demand_total.xlsx'
output_file = 'POD_Demand.xlsx'

distribute_demand_by_time(total_demand_file, output_file)


✅ Distributed demand file generated: POD_Demand.xlsx


# Integrate all demands

In [10]:
def combine_demand_data(shelter_file, pod_file, output_file, num_scenarios=100):
    # Read data from both files (assuming data is in each scenario sheet)
    shelter_data = pd.read_excel(shelter_file, sheet_name=None)
    pod_data = pd.read_excel(pod_file, sheet_name=None)

    # Create an output Excel writer using openpyxl
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        for scenario_num in range(num_scenarios):
            scenario_sheet = f'Scenario_{scenario_num}'

            # Get data for the current scenario
            shelter_scenario = shelter_data.get(scenario_sheet, pd.DataFrame())
            pod_scenario = pod_data.get(scenario_sheet, pd.DataFrame())

            # Initialize combined data structure
            combined_data = pd.DataFrame(0.0, index=range(43), columns=range(12))  # Initialize with float values

            # Assign data to the correct time periods and nodes
            if not shelter_scenario.empty:
                combined_data.iloc[6:27, 0:6] = shelter_scenario.iloc[:, 1:7].values  # 21 shelters, periods 0-5

            if not pod_scenario.empty:
                combined_data.iloc[36:43, 6:12] = pod_scenario.iloc[:, 1:7].values  # 7 PODs, periods 6-11

            # Add index column
            combined_data.insert(0, 'Index', range(1, 44))

            # Write combined data to a new sheet
            combined_data.to_excel(writer, sheet_name=scenario_sheet, index=False)

    print(f"✅Combined data saved to {output_file}")

# Example usage:
combine_demand_data('Shelter_Demand.xlsx', 'POD_Demand.xlsx', 'demand_data_original.xlsx')

✅Combined data saved to demand_data_original.xlsx


# Specify the instance version here

In [12]:
# Specify the instance version
instance_version = "original" #Versions: "original", "smallsize", "mediumsize", "largesize"

# Define file names dynamically based on instance version
location_file = f"fullDist Set V Data Location-{instance_version}.xlsx"
county_file = f"fullDist Set C Data-{instance_version}.xlsx"
resource_file = f"fullDist R-{instance_version}.xlsx"
time_matrix_file = f"durationFull-{instance_version}.txt"
distance_matrix_file = f"distanceFull-{instance_version}.txt"
inputs_file = f"fullDistInputs-{instance_version}.xlsx"
k_ijt_file = f"K_ijt_data_fixed-{instance_version}.xlsx"

# Specify the number of scenarios here

In [14]:
# Define the number of scenarios
num_sce = 5 # Example: 10 scenarios (adjust as needed)
sce = list(range(num_sce)) 

# Generate equal probabilities for each scenario
P = [1/num_sce] * num_sce

#Demand data
demand_file = f"demand_data_{instance_version}.xlsx"

# Dynamically generate sheet names based on the number of scenarios
sheet_names = [f"Scenario_{i}" for i in sce]

print(f"Number of scenarios: {num_sce}")
print(f"Generated sheet names: {sheet_names}")

Number of scenarios: 5
Generated sheet names: ['Scenario_0', 'Scenario_1', 'Scenario_2', 'Scenario_3', 'Scenario_4']


In [15]:
# Location Data
df_V = pd.read_excel(location_file)
print(df_V.head())

# County Data
df_C = pd.read_excel(county_file)
print(df_C.head())

# Resource Data
df_R = pd.read_excel(resource_file)
print(df_R.head())

# Time and Distance Matrices
timeMatrix = np.loadtxt(time_matrix_file)  # Time duration matrix (seconds)
distanceMatrix = np.loadtxt(distance_matrix_file)  # Distance matrix
print(timeMatrix)
print(distanceMatrix)

# Inputs Data
df_inputs = pd.read_excel(inputs_file)
print(df_inputs.head())

# K_ijt Data
df_K = pd.read_excel(k_ijt_file)
print(df_K.head())

   Index County       Type  Cluster                         Facility  \
0      0    NaN  Warehouse      NaN        SCEMD Winnsboro Warehouse   
1      1    NaN  Warehouse      4.0   US Military Base - Cot Storage   
2      2    NaN   Supplier      NaN                   SOPAKCO (Food)   
3      3    NaN   Supplier      NaN                   Lipsey (Water)   
4      4    NaN   Supplier      NaN  FEMA Warehouse (Food and Water)   

                                             Address   latitude  longitude  \
0              100 Tidewater Dr, Winnsboro, SC 29180 -81.098402  34.403608   
1                Garners Ferry Rd, Hopkins, SC 29061 -80.833969  34.045263   
2                118 S Cypress St, Mullins, SC 29574 -79.262558  34.202395   
3               1701 Oakbrook Dr, Norcross, GA 30093 -84.196174  33.913660   
4  3780 Southside Industrial Pkwy, Atlanta, GA 30354 -84.373590  33.650208   

   Shelter Capacity  Pallet Capacity  ...  P-Kit StartInv  Water StartInv  \
0               NaN  

In [16]:
# Load demand data
d_its = {} # Dictionary to store demand data in (location, time, scenario) format

for s, sheet in enumerate(sheet_names):  # Scenario index starts from 0
    df = pd.read_excel(demand_file, sheet_name=sheet, header=0, index_col=0) # Read demand data for the current sheet
    
    # Replace NaN values with 0 (if applicable)
    df = df.fillna(0)
    # Convert DataFrame to a NumPy array for iteration
    demand_array = df.values
    
    # Process demand data for each location and time period
    for location_index in range(demand_array.shape[0]): #Rows represent locations
        for time_period in range(demand_array.shape[1]): #Columns represent time periods
            d_its[(location_index, time_period, s)] = demand_array[location_index, time_period]

# Print a sample to verify the correct location
print("Sample demand data for location 5, time 2, scenario 0:", d_its[(5, 2, 0)])
print("Sample demand data for location 6, time 2, scenario 0:", d_its[(6, 2, 0)])

Sample demand data for location 5, time 2, scenario 0: 0.0
Sample demand data for location 6, time 2, scenario 0: 51.82724118696117


In [17]:
# Defining sets

# County Set
C = len(df_C)
print('All counties:', C)

# Location Sets
V = len(df_V)
print('All locations:', V)

# Subsets of Location Set (V0, V1, V2, V3)
V0 = []  # Warehouses and Suppliers
V1 = []  # Shelters
V2 = []  # RSAs (Staging Areas)
V3 = []  # PoDs (Points of Distribution)


for i in range(V):
    if df_V['Type'][i] == 'Warehouse' or df_V['Type'][i] == 'Supplier':
        V0.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'Shelter':
        V1.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'RSA':
        V2.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'PoD':
        V3.append(df_V['Index'][i])

# Print lengths of subsets
print('Warehouse/Supplier locations:', len(V0), V0)
print('Shelter locations:', len(V1), V1)
print('RSA locations:', len(V2), V2)
print('PoD locations:', len(V3), V3)

# Commodity Set
R = len(df_R)
print('All resources:', R)

# Time Period Set
T = int(df_inputs['Hours in 6 days'][0] / tperiod)
print('Time periods:', T)

#Time Period Subsets
T_Bef = list(range(int(T / 2)))
T_Aft = list(range(int(T / 2), T))
print('Time Before Landfall:', T_Bef)
print('Time After Landfall:', T_Aft)

# Print to verify
print("Scenario indices:", sce)
print("Scenario probabilities:", P)

All counties: 7
All locations: 43
Warehouse/Supplier locations: 6 [0, 1, 2, 3, 4, 5]
Shelter locations: 21 [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
RSA locations: 9 [27, 28, 29, 30, 31, 32, 33, 34, 35]
PoD locations: 7 [36, 37, 38, 39, 40, 41, 42]
All resources: 6
Time periods: 12
Time Before Landfall: [0, 1, 2, 3, 4, 5]
Time After Landfall: [6, 7, 8, 9, 10, 11]
Scenario indices: [0, 1, 2, 3, 4]
Scenario probabilities: [0.2, 0.2, 0.2, 0.2, 0.2]


In [18]:
#Create Arc Set where cannot go from shelter to PoD

A = set() #set to store arc pairs

no_demand=[]
for i in set(V1).union(set(V3)):
    i_has_demand = any(df_V['County'][i] == df_C['County'][c] and df_C['Shelter Total Demand'][c] > 0 for c in range(C))
    if not i_has_demand:
        no_demand.append(i)
print(no_demand)

for i in range(V):
    for j in range(V):
        if i != j and i not in no_demand and j not in no_demand:
            if (i not in V1 or j not in V3) and (i not in V3 or j not in V1):
                A.add((i, j))


print("Number of arcs:", len(A))
print("Number of arcs eliminated by this arc set:", (V*V)-len(A))

[23, 42]
Number of arcs: 1400
Number of arcs eliminated by this arc set: 449


In [19]:
#Defining parameters
W_k = df_inputs['People on Truck'][0] #number of people on a truck
tp_salary = df_inputs['People on Truck'][0]*tperiod; # salary per time period 
fuel = df_inputs['Fuel Cost Per Mile'][0]*0.000621371; # fuel cost per meter
open_array = [df_inputs['Open Cost V0'][0],df_inputs['Open Cost V1'][0],df_inputs['Open Cost V2'][0],df_inputs['Open Cost V3'][0]] #Assumed opening costs

# Initialize inventory list
Inv = []

# Define starting inventory based on instance version
if instance_version == "original":
    for i in range(V):
        Inv.extend([
            df_V['S-Kit StartInv'][i],
            df_V['P-Kit StartInv'][i],
            df_V['Water StartInv'][i],
            df_V['Food StartInv'][i],
            df_V['Cot StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]

elif instance_version == "largesize":
    for i in range(V):
        Inv.extend([
            df_V['S-Kit StartInv'][i],
            df_V['P-Kit StartInv'][i],
            df_V['Water StartInv'][i],
            df_V['Food StartInv'][i],
            df_V['Cot StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]

elif instance_version == "smallsize":
    for i in range(V):
        Inv.append(df_V['Water StartInv'][i])
    I_ir0 = [[inv] for inv in Inv]

elif instance_version == "mediumsize":
    for i in range(V):
        Inv.extend([
            df_V['Water StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]


#penalty cost for location i p_i[I]
p_i = df_V['P_i'].tolist()  # Convert the column directly to a list

#time periods to travel i,j t_ijt[i][j]
seconds = 3600
t_ijt = [[math.ceil((timeMatrix[i][j] / seconds) / tperiod) for j in range(V)] for i in range(V)]

#truck delay h_i[i]
h_i=[]
for i in range(V):
    h_i.append(df_V['TruckDelay'][i])

#cost to travel i,j m_ij[i][j]
m_ij=[]
for i in range(V):
    row=[]
    for j in range(V):
        m_ij_value=((distanceMatrix[i][j]*fuel)+(t_ijt[i][j]*tp_salary*W_k)+(h_i[i]*tp_salary*W_k))
        row.append(m_ij_value)
    m_ij.append(row)

#Cost to procure resource r from location i cost_ir[i][r]
cost=[]
for i in V0:
    cost.extend([0,0,df_V['Water PC'][i],df_V['Food PC'][i],0,0])
    cost_ir = [cost[i:i+R] for i in range(0, len(cost), R)]

#Fraction of a truck 1 pallet of resource r requires w_r[R]
w_r = [1 / pallet for pallet in df_R['Pallet Units on Truck']]
    
#Factor of pallet required per commodity r of kit g_shelter[R]
g_shelter = df_R['pallet per S-Kit'].tolist()

#Factor of pallet required per commodity r of kit g_pod[R]
g_pod = df_R['pallet per P-Kit'].tolist()

# Initialize K_ijt as a nested dictionary {i: {j: {t: capacity}}}
K_ijt = {}

# Populate K_ijt with data from df_K
for index, row in df_K.iterrows():
    i, j, t, capacity = row['From_Node'], row['To_Node'], row['Time_Period'], row['Capacity']
    K_ijt.setdefault(i, {}).setdefault(j, {})[t] = capacity

# Defining Master Problem

In [21]:
# Create the master problem model
master_model = gp.Model("MasterProblem")

# Define the inventory decision variables I[i, r, t] for first-stage decisions
# These represent the inventory of commodity r at location i at the start of time period t
I = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            I[i, r, t] = master_model.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"I_{i}_{r}_{t}")


# Define the auxiliary variable θ with an initial lower bound, used for Benders cuts
theta = master_model.addVar(vtype=GRB.CONTINUOUS, lb = thetaLB, name="theta")

for i in range(V):
    for r in range(R):
        master_model.addConstr(I[i, r, 0] == I_ir0[i][r], name=f"initial_inventory_{i}_{r}")

# Add constraints to ensure inventory at warehouses only decreases over time
for i in V0:
    for r in range(R):
        for t in range(1, T):
            master_model.addConstr(I[i, r, t] <= I[i, r, t-1], name=f"inventory_decrease_warehouse_{i}_{r}_{t}")

# Add constraints to ensure inventory at PoDs only increases before landfall
for i in V3:
    for r in range(R):
        for t in T_Bef[1:]:
            master_model.addConstr(I[i, r, t] >= I[i, r, t-1], name=f"inventory_increase_pod_{i}_{r}_{t}")

# Add constraints for the lower bound of inventory at warehouses (V0)
for i in V0:  # Loop over warehouse nodes
    for r in range(R):  # Loop over commodities
        for t in range(1, T):  # Loop over time periods (from t=1 to T-1)
            # Calculate the total arc capacity for the current time period
            arc_capacity = sum(K_ijt[i][j][t] for j in range(V) if (i, j) in A)
            # Calculate max outflow for commodity r
            max_outflow_r = arc_capacity  / w_r[r]
            # Check if the initial inventory is greater than or equal to max_outflow_r
            if I_ir0[i][r] >= max_outflow_r:
                # Add the lower bound constraint
                master_model.addConstr(I[i, r, t] >= I_ir0[i][r] - max_outflow_r, name=f"LowerBound_Warehouse_{i}_{r}_{t}")


# Add constraints for the upper bound of inventory at PoDs (V3)
for i in V3:  # PoDs
    for r in range(R):  # Commodities
        for t in T_Bef[1:]:  # Time periods before landfall, excluding t=0
            # Calculate the total arc capacity for the current time period
            arc_capacity = sum(K_ijt[j][i][t] for j in range(V) if (j, i) in A)
            # Calculate max inflow for commodity r
            max_inflow_r = arc_capacity / w_r[r]            
            # Add the upper bound constraint
            master_model.addConstr(I[i, r, t] <= I_ir0[i][r] + max_inflow_r, name=f"UpperBound_PoD_{i}_{r}_{t}")

# Set the objective function to minimize θ (representing the expected second-stage cost)
master_model.setObjective(theta, GRB.MINIMIZE)

# Set output flag to 0 to avoid printing Gurobi logs
master_model.setParam("OutputFlag", 0);

# Update the model to integrate new variables
master_model.update()

Set parameter Username
Set parameter LicenseID to value 2611420
Academic license - for non-commercial use only - expires 2026-01-17


# Defining Optimality Subproblem

In [23]:
# Define the optimality subproblem
sub_opt = gp.Model("subprob_optimality")
sub_opt.modelSense = GRB.MINIMIZE

# Define second-stage decision variables U, S, and f for the optimality subproblem
U_opt = {}
S_opt = {}
f_opt = {}

# U: Demand shortage at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        U_opt[i, t] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"U_opt_{i}_{t}")

# S: Demand satisfied at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        S_opt[i, t] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"S_opt_{i}_{t}")

# f: Quantity of commodity r traveling on link (i, j) at time period t in scenario s
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for r in range(R):
                for t in range(T):
                    f_opt[i, j, r, t] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"f_opt_{i}_{j}_{r}_{t}")

# Objective: Minimize the expected demand shortage cost plus transportation cost
sub_opt.setObjective(
    gp.quicksum(
        p_i[i] * U_opt[i, t]  # Demand shortage cost
        for i in range(V) for t in range(T)) +
    # Term 1: For nodes in V0 (Procurement + Transportation Costs)
    gp.quicksum(
        (cost_ir[i][r] + m_ij[i][j]) * f_opt[i, j, r, t]
        for i in V0 for j in range(V) if (i, j) in A
        for r in range(R) for t in range(T)) +
    # Term 2: For nodes not in V0 (Transportation Cost only)
    gp.quicksum(
        m_ij[i][j] * f_opt[i, j, r, t]
        for i in [i for i in range(V) if i not in V0]
        for j in range(V) if (i, j) in A
        for r in range(R) for t in range(T)))


# Constraint for inventory flow balance with U and S
pi_opt_inventory_balance_U = {}
for i in range(V):
    for t in range(1, T):  # Start from t=1 since t=0 is the initial condition
        pi_opt_inventory_balance_U[i, t] = sub_opt.addConstr(U_opt[i, t] - U_opt[i, t-1] + S_opt[i, t] == 0, name=f"inventory_balance_U_opt_{i}_{t}")

# Initial shortage for demand at t=0
pi_opt_initial_shortage_U = {}
for i in range(V):
    pi_opt_initial_shortage_U[i] = sub_opt.addConstr(U_opt[i, 0] == 0, name=f"initial_shortage_U_opt_{i}")

# Flow conservation constraint for f with the RHS set to 0 temporarily, using appropriate g_shelter or g_pod
pi_opt_flow_conservation = {}
for i in range(V):
    for r in range(R):
        for t in range(T - 1):
            # Add the constraint to the model
            pi_opt_flow_conservation[i, r, t] = sub_opt.addConstr(
                -gp.quicksum(f_opt[i, j, r, t] for j in range(V) if (i, j) in A)
                + gp.quicksum(f_opt[j, i, r, t - t_ijt[j][i] + 1] for j in range(V) if (i, j) in A and t - t_ijt[j][i] + 1 >= 0)
                - S_opt[i, t] * (g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0) == 0,
                name=f"flow_conservation_opt_{i}_{r}_{t}")

# Arc capacity constraint
pi_opt_arc_capacity = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                    pi_opt_arc_capacity[i, j, t] = sub_opt.addConstr(-gp.quicksum(w_r[r] * f_opt[i, j, r, t] for r in range(R)) >= -K_ijt[i][j][t],
                        name=f"arc_capacity_opt_{i}_{j}_{t}")
"""

# Define parameters
truck_capacity = 256  # Capacity of one truck
num_trucks = 12       # Number of available trucks

# Flow capacity constraint for optimality subproblem, summing over all valid arcs
for t in range(T):
    pi_opt_arc_capacity = sub_opt.addConstr(
        -gp.quicksum(w_r[r] * f_opt[i, j, r, t] for i in range(V) for j in range(V) if (i, j) in A for r in range(R)) >= -num_trucks * truck_capacity,
        name=f"arc_capacity_opt_{t}"
    )
"""

sub_opt.setParam("OutputFlag", 0);

# Defining Feasibility Subproblem

In [25]:
# Define the feasibility subproblem
sub_feas = gp.Model("subprob_feasibility")
sub_feas.modelSense = GRB.MINIMIZE

# Define second-stage decision variables U, S, and f for the feasibility subproblem
U_feas = {}
S_feas = {}
f_feas = {}

# U: Demand shortage at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        U_feas[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"U_feas_{i}_{t}")

# S: Demand satisfied at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        S_feas[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"S_feas_{i}_{t}")

# f: Quantity of commodity r traveling on link (i, j) at time period t in scenario s
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for r in range(R):
                for t in range(T):
                    f_feas[i, j, r, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"f_feas_{i}_{j}_{r}_{t}")

# Now define slack variables ε (epsilon) for each group in the objective function
# ε variables with objective coefficients corresponding to their respective constraint group

epsilon_1 = {}
for i in range(V):
    for r in range(R):
        for t in range(T-1):
            epsilon_1[i, r, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon1_{i}_{r}_{t}")

epsilon_2 = {}
for i in range(V):
    for r in range(R):
        for t in range(T-1):
            epsilon_2[i, r, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon2_{i}_{r}_{t}")

epsilon_3 = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for t in range(T):
                epsilon_3[i, j, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon3_{i}_{j}_{t}")
"""
epsilon_3 = {}
for t in range(T):
    epsilon_3[t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon3_{t}")
"""
sub_feas.setObjective(
    gp.quicksum(epsilon_1[i, r, t] + epsilon_2[i, r, t] for i in range(V) for r in range(R) for t in range(T - 1)) +
    gp.quicksum(epsilon_3[i, j, t] for i in range(V) for j in range(V) if (i, j) in A for t in range(T)))

# Constraint: Inventory flow balance for U, S, and epsilon variables
pi_feas_inventory_balance_U = {}
for i in range(V):
    for t in range(1, T):  # Note: starting from t=1 as per the constraint
        pi_feas_inventory_balance_U[i, t] = sub_feas.addConstr(
            U_feas[i, t] - U_feas[i, t-1] + S_feas[i, t] == 0,
            name=f"inventory_balance_U_feas_{i}_{t}")

# Initialize shortage for U at t=0
pi_feas_initial_shortage_U = {}
for i in range(V):
    pi_feas_initial_shortage_U[i] = sub_feas.addConstr(U_feas[i, 0] == 0, name=f"initial_shortage_U_feas_{i}")

# Flow conservation constraint
pi_feas_flow_conservation = {}
for i in range(V):
    for r in range(R):
        for t in range(T - 1):
            pi_feas_flow_conservation[i, r, t] = sub_feas.addConstr(
                -gp.quicksum(f_feas[i, j, r, t] for j in range(V) if (i, j) in A)
                + gp.quicksum(f_feas[j, i, r, t - t_ijt[j][i] + 1] for j in range(V) if (i, j) in A and t - t_ijt[j][i] + 1 >= 0)
                - S_feas[i, t] * (g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0)
                + epsilon_1[i, r, t] - epsilon_2[i, r, t] == 0,
                name=f"flow_conservation_feas_{i}_{r}_{t}")

# Flow capacity constraint for all links, time periods, and scenarios
pi_feas_arc_capacity = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                pi_feas_arc_capacity[i, j, t] = sub_feas.addConstr(
                    -gp.quicksum(w_r[r] * f_feas[i, j, r, t] for r in range(R))
                    + epsilon_3[i, j, t] >= -K_ijt[i][j][t],
                    name=f"arc_capacity_feas_{i}_{j}_{t}")
"""
# Define parameters
truck_capacity = 256  # Capacity of one truck
num_trucks = 12       # Number of available trucks

# Flow capacity constraint for the feasibility subproblem
pi_feas_arc_capacity = {}
for t in range(T):
    pi_feas_arc_capacity[t] = sub_feas.addConstr(
        -gp.quicksum(w_r[r] * f_feas[i, j, r, t] for i in range(V) for j in range(V) if (i, j) in A for r in range(R)) 
        + epsilon_3[t] 
        >= -num_trucks * truck_capacity,
        name=f"arc_capacity_feas_{t}"
    )
"""
sub_feas.setParam("OutputFlag", 0);

# Creating Benders Loop

In [27]:
LB = thetaLB
UB = 1e10

Ocuts = 0
Fcuts = 0

iter = 0
eps = 1e-5  # tolerance parameter
eps1 = 1e-12
# Number of scenarios
num = len(sce)

# First, solve the master problem to get the values of first-stage variables and theta
I_values = {}  # Store I values for each (i, r, t)
theta_value = 0  # Initialize theta value

# Optimality subproblem dual multipliers
pi_opt_inventory_balance_U_val = {s: {} for s in range(num)}
pi_opt_initial_shortage_U_val = {s: {} for s in range(num)}
#pi_opt_demand_satisfaction_val = {s: {} for s in range(num)}
pi_opt_flow_conservation_val = {s: {} for s in range(num)}
#pi_opt_inventory_balance_val = {s: {} for s in range(num)}
pi_opt_arc_capacity_val = {s: {} for s in range(num)}
    
# Feasibility subproblem dual multipliers
pi_feas_inventory_balance_U_val = {}
pi_feas_initial_shortage_U_val = {}
#pi_feas_demand_satisfaction_val = {}
pi_feas_flow_conservation_val = {}
#pi_feas_inventory_balance_val = {}
pi_feas_arc_capacity_val = {}

while (UB - LB) / max(1e-10,abs(UB)) > eps:
    iter += 1

    # Optimize the master problem
    master_model.optimize()

    # Check if the optimization was successful
    if master_model.status == GRB.OPTIMAL:
        # Extract the values of first-stage decision variables I[i, r, t]
        for i in range(V):
            for r in range(R):
                for t in range(T):
                    I_values[i, r, t] = I[i, r, t].x

        # Get the value of theta
        theta_value = theta.x

        # Update the lower bound
        LB = master_model.objVal       
         
    elif master_model.status == GRB.INFEASIBLE:
        print("Master problem is infeasible.")
        exit(0)  # Or handle infeasibility as required for your problem

    elif master_model.status == GRB.UNBOUNDED:
        print("Master problem is unbounded.")
        exit(0)  # Or handle unboundedness as required for your problem

    else:
        print(f"Optimization was stopped with status {master_model.status}")
        exit(0)
    

    # Initialize a vector to store the optimal objective value for the optimality subproblem for each scenario
    Q_opt = np.zeros(num)
    # Initialize a vector to store the optimal objective value for the feasibility subproblem for each scenario
    w_feas = np.zeros(num)
    FLAG = 0; #(0 if I_hat is feasible ∀ s, and 1 otherwise)

    # Update RHS for optimality subproblem constraints for every scenario
    infeasible_sce = -1
    for s in range(num):  # Loop over all scenarios
        # Update RHS of inventory flow balance constraints
        for i in range(V):
            for t in range(1, T):
                pi_opt_inventory_balance_U[i, t].setAttr(GRB.Attr.RHS, d_its[(i, t, s)])
        
        # Update RHS of initial shortage constraints
        for i in range(V):
            pi_opt_initial_shortage_U[i].setAttr(GRB.Attr.RHS, d_its[(i, 0, s)])
        
        # Update RHS of flow conservation constraints
        for i in range(V):
            for r in range(R):
                for t in range(T - 1):
                    pi_opt_flow_conservation[i, r, t].setAttr(GRB.Attr.RHS, I_values[i, r, t+1] - I_values[i, r, t])
        """
        # Update RHS of inventory balance constraints
        for i in range(V):
            for r in range(R):
                for t in range(T):
                    pi_opt_inventory_balance[i, r, t].setAttr(GRB.Attr.RHS, -I_values[i, r, t])
        """
        sub_opt.optimize()    
        if sub_opt.status == GRB.OPTIMAL:
            # Update the optimal objective value for each scenario
            Q_opt[s] = sub_opt.objVal
    
            # Retrieve and store dual values for each relevant constraint
            for key, constraint in pi_opt_inventory_balance_U.items():
                pi_opt_inventory_balance_U_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_initial_shortage_U.items():
                pi_opt_initial_shortage_U_val[s][key] = constraint.pi
        
            #for key, constraint in pi_opt_demand_satisfaction.items():
             #   pi_opt_demand_satisfaction_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_flow_conservation.items():
                pi_opt_flow_conservation_val[s][key] = constraint.pi
        
            #for key, constraint in pi_opt_inventory_balance.items():
             #   pi_opt_inventory_balance_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_arc_capacity.items():
                pi_opt_arc_capacity_val[s][key] = constraint.pi
        
        elif sub_opt.status == GRB.INFEASIBLE:
            # Set FLAG to 1 if infeasible, indicating the need for a feasibility cut
            FLAG = 1
            infeasible_sce = s
            break;
        else:
            # Handle other statuses
            print('One of the scenario subproblems is infeasible')
            print('status =', sub_opt.status)
            exit(0)
    print("FLAG =", FLAG)
    if FLAG == 1:
        
            # Update RHS for feasibility subproblem constraints for every scenario
            s = infeasible_sce
            
            # Update RHS of inventory flow balance constraints for U
            for i in range(V):
                for t in range(1, T):
                    pi_feas_inventory_balance_U[i, t].setAttr(GRB.Attr.RHS, d_its[(i, t, s)])
            
            # Update RHS of initial shortage constraints for U at t=0
            for i in range(V):
                pi_feas_initial_shortage_U[i].setAttr(GRB.Attr.RHS, d_its[(i, 0, s)])
            
            
            # Update RHS of flow conservation constraints
            for i in range(V):
                for r in range(R):
                    for t in range(T - 1):
                        pi_feas_flow_conservation[i, r, t].setAttr(GRB.Attr.RHS, I_values[i, r, t+1] - I_values[i, r, t])
                   
            # Optimize the feasibility subproblem
            sub_feas.optimize()
            w_feas[s] = sub_feas.objVal  # Store the optimal objective value for this scenario
            #print("sub_feas.objval = ", sub_feas.objVal)

            # Retrieve and store dual values for each relevant constraint in the feasibility subproblem
            for key, constraint in pi_feas_inventory_balance_U.items():
                pi_feas_inventory_balance_U_val[key] = constraint.pi
            for key, constraint in pi_feas_initial_shortage_U.items():
                pi_feas_initial_shortage_U_val[key] = constraint.pi
            #for key, constraint in pi_feas_demand_satisfaction.items():
             #   pi_feas_demand_satisfaction_val[key] = constraint.pi            
            for key, constraint in pi_feas_flow_conservation.items():
                pi_feas_flow_conservation_val[key] = constraint.pi            
            #for key, constraint in pi_feas_inventory_balance.items():
             #   pi_feas_inventory_balance_val[key] = constraint.pi            
            for key, constraint in pi_feas_arc_capacity.items():
                pi_feas_arc_capacity_val[key] = constraint.pi   

            if w_feas[s] > eps1:
                # Initialize the feasibility cut expression
                feasibility_cut_expr = gp.LinExpr()
            
                # Add the terms for inventory flow balance for U and S constraints
                for key, dual_val in pi_feas_inventory_balance_U_val.items():
                    i, t = key
                    feasibility_cut_expr += dual_val * d_its[(i, t, s)]
                    
                # Add the terms for initial shortage for U constraints
                for key, dual_val in pi_feas_initial_shortage_U_val.items():
                    i = key
                    feasibility_cut_expr += dual_val * d_its[(i, 0, s)]
                
                # Add the terms for demand satisfaction constraints
                #for key, dual_val in pi_feas_demand_satisfaction_val.items():
                #    i, t = key
                #    feasibility_cut_expr += dual_val * 0

                # Add the terms for flow conservation constraints
                for key, dual_val in pi_feas_flow_conservation_val.items():
                    i, r, t = key
                    if abs(dual_val) > 1e-7:
                        feasibility_cut_expr += dual_val * (I[i, r, t+1] - I[i, r, t])
                
                # Add the terms for arc capacity constraints
                for key, dual_val in pi_feas_arc_capacity_val.items():
                    i, j, t = key
                    feasibility_cut_expr += dual_val * -K_ijt[i][j][t]

                """
                for key, dual_val in pi_feas_arc_capacity_val.items():
                    t = key
                    feasibility_cut_expr += dual_val * (-num_trucks * truck_capacity)
                """
                # Add the feasibility cut to the master problem
                master_model.addConstr(feasibility_cut_expr <= 0, name=f"feasibility_cut_scenario_{s}")
                Fcuts += 1  # Increment the counter for feasibility cuts                

    else:
        print("here")
        #print("Q_opt = ", Q_opt)
        #print("w_feas =", w_feas)
        # Calculate the temporary upper bound using only the second-stage costs
        tempUB = sum(Q_opt[s] * P[s] for s in range(num))        
        if tempUB < UB:
            UB = tempUB
        if (UB - LB) / max(1e-10, abs(LB)) < eps:
            break

        else:
            # Initialize the optimality cut expression
            optimality_cut_expr = gp.LinExpr()
            for s in range(num):
                 
                # Add the terms for inventory flow balance with U and S constraints
                for key, dual_val in pi_opt_inventory_balance_U_val[s].items():
                    i, t = key
                    optimality_cut_expr += P[s] * dual_val * d_its[(i, t, s)]
                
                # Add the terms for initial shortage for U constraints
                for key, dual_val in pi_opt_initial_shortage_U_val[s].items():
                    i = key
                    optimality_cut_expr += P[s] * dual_val * d_its[(i, 0, s)]
                
                # Add the terms for demand satisfaction constraints
                #for key, dual_val in pi_opt_demand_satisfaction_val[s].items():
                #    i, t = key
                #    optimality_cut_expr += P[s] * dual_val * 0
                
                # Add the terms for flow conservation constraints
                for key, dual_val in pi_opt_flow_conservation_val[s].items():
                    i, r, t = key
                    if abs(dual_val) > 1e-7:
                        optimality_cut_expr += P[s] * dual_val * (I[i, r, t + 1] - I[i, r, t])

                # Add the terms for arc capacity constraints
                for key, dual_val in pi_opt_arc_capacity_val[s].items():
                    i, j, t = key
                    optimality_cut_expr += P[s] * dual_val * -K_ijt[i][j][t]
                """
                for key, dual_val in pi_opt_arc_capacity_val[s].items():
                    t = key
                    optimality_cut_expr += P[s] * dual_val * (-num_trucks * truck_capacity)
                """
            # Add the optimality cut to the master model
            master_model.addConstr(theta >= optimality_cut_expr, name="optimality_cut")
            Ocuts += 1  # Increment the counter for optimality cuts

            

    # Print information at each iteration
    print("iter =", iter)
    print("LB =", LB)
    print("UB =", UB)
    print("===============================================================")


# Final summary after convergence
print("#########################################################################################")
print("#########################################################################################")
print("Final iteration =", iter)
print("Number of feasibility cuts added =", Fcuts)
print("Number of optimality cuts added =", Ocuts)
print("Lower Bound (LB) =", LB)
print("Upper Bound (UB) =", UB)
print("FLAG =", FLAG)

end_time = time.time()  # Record the end time

execution_time = end_time - start_time  # Calculate the execution time
print(f"Execution Time: {execution_time} seconds")

FLAG = 1
iter = 1
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 2
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 3
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 4
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 5
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 6
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 7
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 8
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 9
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 10
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 11
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 12
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 13
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 14
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 15
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 16
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 17
LB = 0.0
UB = 10000000000.0
FLAG = 1
iter = 18
LB = 0.0
UB = 10000000000.0
FLAG = 0
here
iter = 19
LB = 0.0
UB = 107489360.58247733
FLAG = 0
here
iter = 20
LB = 0.0
UB = 98397352.37626065
FLAG = 1
iter = 21
LB = 0.0
UB = 98397352.37626065

KeyboardInterrupt: 